In [2]:
import pandas as pd

# ---------------------------------------------------------------------
# 1. Data sources (verified working OWID grapher CSV export endpoints)
# ---------------------------------------------------------------------
HAPPINESS_URL = (
    "https://ourworldindata.org/grapher/happiness-cantril-ladder.csv"
    "?v=1&csvType=full&useColumnShortNames=true"
)
GDP_URL = (
    "https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv"
    "?v=1&csvType=full&useColumnShortNames=true"
)

# Add more World Bank-sourced OWID indicators here if you want to enrich
# further later, e.g.:
# UNEMPLOYMENT_URL = "https://ourworldindata.org/grapher/unemployment-rate.csv?v=1&csvType=full&useColumnShortNames=true"
# GINI_URL = "https://ourworldindata.org/grapher/economic-inequality-gini-index.csv?v=1&csvType=full&useColumnShortNames=true"

# A minimal set of OWID aggregate codes to exclude -- OWID includes rows for
# continents, income groups, and world totals alongside actual countries.
# We drop anything whose Code starts with these prefixes so the model only
# trains on real countries.
AGGREGATE_PREFIXES = ("OWID_", "WB_")


def load_and_clean(url: str, new_name: str) -> pd.DataFrame:
    """
    OWID's grapher CSV export is not fully consistent about column naming
    (some indicators get a short machine name, some fall back to the full
    descriptive title, and letter case varies). Rather than hardcode a
    column name that might not match, we rely on the fact that every
    export has the SAME first three columns in the SAME order:
    entity/Entity, code/Code, year/Year -- followed by exactly one value
    column (sometimes a trailing region column too, which we ignore).
    """
    df = pd.read_csv(
        url,
        storage_options={"User-Agent": "Mozilla/5.0 (research data fetch)"},
    )
    df.columns = [c.strip() for c in df.columns]
    # First three columns are always Country / Code / Year, in that order
    country_col, code_col, year_col = df.columns[0], df.columns[1], df.columns[2]
    value_col = df.columns[3]  # the 4th column is always the indicator value

    df = df[[country_col, code_col, year_col, value_col]]
    df.columns = ["Country", "Code", "Year", new_name]

    # Drop continent / income-group / world aggregate rows
    df = df[~df["Code"].astype(str).str.startswith(AGGREGATE_PREFIXES)]
    df = df.dropna(subset=["Code"])
    return df


def main():
    print("Downloading World Happiness Report (Cantril Ladder) data...")
    happiness = load_and_clean(HAPPINESS_URL, "HappinessScore")

    print("Downloading World Bank GDP per capita data...")
    gdp = load_and_clean(GDP_URL, "GDPPerCapita_WB")

    print("Merging on Country code + Year...")
    merged = pd.merge(
        happiness, gdp, on=["Code", "Year"], how="inner", suffixes=("", "_gdp")
    )
    # Keep a single clean Country column
    merged = merged[["Country", "Code", "Year", "HappinessScore", "GDPPerCapita_WB"]]
    merged = merged.sort_values(["Country", "Year"]).reset_index(drop=True)

    # Restrict to the WHR's usual reporting window; adjust as needed
    merged = merged[(merged["Year"] >= 2011) & (merged["Year"] <= 2024)]

    out_path = "whr_worldbank_merged.csv"
    merged.to_csv(out_path, index=False)
    print(f"Saved {len(merged)} rows covering {merged['Country'].nunique()} "
          f"countries to {out_path}")
    print(merged.head(10))


if __name__ == "__main__":
    main()

Merging on Country code + Year...
Saved 1897 rows covering 159 countries to whr_worldbank_merged.csv
       Country Code  Year  HappinessScore  GDPPerCapita_WB
0  Afghanistan  AFG  2011          4.2580        2757.0525
1  Afghanistan  AFG  2012          4.0400        2985.3190
2  Afghanistan  AFG  2014          3.5750        3017.9426
3  Afghanistan  AFG  2015          3.3600        2967.6921
4  Afghanistan  AFG  2016          3.7940        2958.7854
5  Afghanistan  AFG  2017          3.6320        2952.9990
6  Afghanistan  AFG  2018          3.2030        2902.3920
7  Afghanistan  AFG  2019          2.5669        2927.2450
8  Afghanistan  AFG  2020          2.5230        2769.6858
9  Afghanistan  AFG  2021          2.4040        2144.1665


In [3]:
"""
Preprocessing pipeline for whr_worldbank_merged.csv

Produces train/val/test splits that are SAFE for the temporal-forecasting
task (no leakage from future years into the training set), plus lag
features for the classical ML baselines (RF, XGBoost) that will later be
stacked with the LSTM.

Run: python preprocess_data.py
Requires: pandas, numpy, scikit-learn, joblib

Outputs (all in ./processed/):
    train.csv, val.csv, test.csv   -- chronologically split, scaled
    scaler.joblib                  -- fitted StandardScaler (fit on TRAIN only)
    data_dictionary.md             -- column-by-column description for handoff
"""

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import joblib

INPUT_FILE = "whr_worldbank_merged.csv"
OUTPUT_DIR = "processed"

# Chronological split -- DO NOT shuffle. This is a forecasting task, so the
# test set must be years the model has never seen, not a random sample.
TRAIN_END_YEAR = 2021   # train on <= 2021
VAL_END_YEAR = 2022      # validate on 2022
# everything after VAL_END_YEAR (2023, 2024) is test

MIN_YEARS_REQUIRED = 8   # drop countries with fewer than this many yearly
                          # observations -- too short a sequence to be
                          # useful for an LSTM, and it thins your data anyway


def audit_missingness(df: pd.DataFrame) -> pd.DataFrame:
    """Print a quick report so you and your teammate both see the same
    picture of data completeness before any cleaning happens."""
    counts = df.groupby("Country")["Year"].nunique().sort_values()
    print(f"\nCountries in raw merged data: {df['Country'].nunique()}")
    print(f"Year range: {df['Year'].min()}–{df['Year'].max()}")
    print(f"Countries with fewer than {MIN_YEARS_REQUIRED} years of data: "
          f"{(counts < MIN_YEARS_REQUIRED).sum()}")
    return counts


def add_lag_features(df: pd.DataFrame, cols_to_lag, n_lags=2) -> pd.DataFrame:
    """
    Adds t-1, t-2, ... lag columns for each requested column, per country.
    These are what the classical ML baselines (RF/XGBoost) will use, since
    unlike the LSTM they can't consume a raw sequence -- they need the
    history flattened into columns.
    """
    df = df.sort_values(["Country", "Year"]).copy()
    for col in cols_to_lag:
        for lag in range(1, n_lags + 1):
            df[f"{col}_lag{lag}"] = df.groupby("Country")[col].shift(lag)
    return df


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(INPUT_FILE)
    audit_missingness(df)

    # --- Step 1: drop countries with too little history ---------------
    year_counts = df.groupby("Country")["Year"].transform("nunique")
    df = df[year_counts >= MIN_YEARS_REQUIRED].copy()

    # --- Step 2: interpolate small internal gaps (not edge gaps) ------
    # A missing year in the MIDDLE of a country's series can be safely
    # interpolated; missing years at the start/end cannot (that would be
    # extrapolation, not interpolation), so we leave those as NaN and drop
    # them at the end.
    df = df.sort_values(["Country", "Year"])
    df["HappinessScore"] = df.groupby("Country")["HappinessScore"].transform(
        lambda s: s.interpolate(method="linear", limit_area="inside")
    )
    df["GDPPerCapita_WB"] = df.groupby("Country")["GDPPerCapita_WB"].transform(
        lambda s: s.interpolate(method="linear", limit_area="inside")
    )
    df = df.dropna(subset=["HappinessScore", "GDPPerCapita_WB"])

    # --- Step 3: transform GDP (heavily right-skewed across countries) -
    # Log transform is standard practice here -- GDP per capita spans
    # ~$500 to ~$130,000, and models trained on the raw scale will be
    # dominated by rich-country variance. This is worth stating explicitly
    # in your methodology section.
    df["log_GDPPerCapita_WB"] = np.log(df["GDPPerCapita_WB"])

    # --- Step 4: lag features for classical ML baselines ---------------
    df = add_lag_features(df, cols_to_lag=["HappinessScore", "log_GDPPerCapita_WB"],
                           n_lags=2)

    # The prediction target: NEXT year's happiness score (this is what
    # makes it a forecasting task rather than a same-year regression)
    df["HappinessScore_target"] = df.groupby("Country")["HappinessScore"].shift(-1)

    # Drop rows where we don't have enough lag history or a target yet
    df = df.dropna(subset=[
        "HappinessScore_lag1", "HappinessScore_lag2",
        "log_GDPPerCapita_WB_lag1", "log_GDPPerCapita_WB_lag2",
        "HappinessScore_target",
    ])

    # --- Step 5: chronological split ------------------------------------
    train = df[df["Year"] <= TRAIN_END_YEAR].copy()
    val = df[df["Year"] == VAL_END_YEAR].copy()
    test = df[df["Year"] > VAL_END_YEAR].copy()

    print(f"\nSplit sizes -- train: {len(train)}, val: {len(val)}, test: {len(test)}")
    print(f"Train years: <= {TRAIN_END_YEAR} | Val year: {VAL_END_YEAR} "
          f"| Test years: > {VAL_END_YEAR}")

    # --- Step 6: scale numeric features -- FIT ON TRAIN ONLY -----------
    # This is the single most common leakage bug in student projects:
    # fitting the scaler on the full dataset before splitting leaks
    # information about the val/test distribution into training.
    feature_cols = [
        "HappinessScore_lag1", "HappinessScore_lag2",
        "log_GDPPerCapita_WB_lag1", "log_GDPPerCapita_WB_lag2",
    ]
    scaler = StandardScaler()
    train[feature_cols] = scaler.fit_transform(train[feature_cols])
    val[feature_cols] = scaler.transform(val[feature_cols])
    test[feature_cols] = scaler.transform(test[feature_cols])

    # --- Step 7: save everything -----------------------------------------
    train.to_csv(os.path.join(OUTPUT_DIR, "train.csv"), index=False)
    val.to_csv(os.path.join(OUTPUT_DIR, "val.csv"), index=False)
    test.to_csv(os.path.join(OUTPUT_DIR, "test.csv"), index=False)
    joblib.dump(scaler, os.path.join(OUTPUT_DIR, "scaler.joblib"))

    write_data_dictionary(feature_cols, df)
    print(f"\nDone. Files saved in ./{OUTPUT_DIR}/")


def write_data_dictionary(feature_cols, df):
    """A short handoff doc so your teammate doesn't have to reverse-engineer
    the pipeline from the code."""
    content = f"""# Data Dictionary -- Processed Happiness Forecasting Dataset

Generated by `preprocess_data.py` from `whr_worldbank_merged.csv`.

## Files
- `train.csv`  -- years <= {TRAIN_END_YEAR}
- `val.csv`    -- year == {VAL_END_YEAR}
- `test.csv`   -- years > {VAL_END_YEAR}
- `scaler.joblib` -- fitted StandardScaler, fit on TRAIN only. Use this same
  scaler object (don't refit) if you need to transform any new data later.

## Columns
| Column | Description |
|---|---|
| Country | Country name |
| Code | ISO3 country code |
| Year | Observation year |
| HappinessScore | Raw Cantril Ladder score (0-10), THIS year |
| GDPPerCapita_WB | Raw GDP per capita (constant international $), THIS year |
| log_GDPPerCapita_WB | Natural log of GDP per capita |
| HappinessScore_lag1 / lag2 | Happiness score 1 / 2 years before, SCALED |
| log_GDPPerCapita_WB_lag1 / lag2 | Log GDP per capita 1 / 2 years before, SCALED |
| HappinessScore_target | **Prediction target**: next year's happiness score (unscaled) |

## Important notes for whoever builds on this
- Features are scaled (StandardScaler, fit on train only). The **target**
  (`HappinessScore_target`) is intentionally left UNSCALED so evaluation
  metrics (RMSE/MAE) are directly interpretable on the original 0-10 scale.
- For the LSTM: use the RAW (unlagged, unscaled) `HappinessScore` and
  `log_GDPPerCapita_WB` sequences per country instead of the lag columns --
  the lag columns are built for the classical ML baselines (RF/XGBoost) and
  for the stacking meta-learner, not for the LSTM itself.
- The scaler was fit ONLY on the training set to avoid leakage. Never refit
  it on val/test.
- Split is chronological (train <= {TRAIN_END_YEAR}, val == {VAL_END_YEAR},
  test > {VAL_END_YEAR}), not random -- this matches the forecasting setup
  and avoids the model "seeing the future" during training.
- Countries with fewer than {MIN_YEARS_REQUIRED} years of data were dropped
  entirely (see console output when you ran this script for the count).
"""
    with open(os.path.join(OUTPUT_DIR, "data_dictionary.md"), "w") as f:
        f.write(content)


if __name__ == "__main__":
    main()


Countries in raw merged data: 159
Year range: 2011–2024
Countries with fewer than 8 years of data: 15

Split sizes -- train: 1127, val: 133, test: 138
Train years: <= 2021 | Val year: 2022 | Test years: > 2022

Done. Files saved in ./processed/
